# DAILY SIMULATION

In [ ]:
# import libraries and formulas
import time
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta

from cpi_hard_method import (
    get_lastest_data,
    get_old_columns,
    parse_old_date,
    append_data,
    truncate_data,
    incremental_data,
    initialize_db,
    db_file,
    file,
)

In [ ]:
raw = pd.read_excel(
    file,
    na_values=["#N/A", "NA", "N/A", ""],
    engine="openpyxl",
)
raw.columns = raw.columns.str.strip()

# change first column name to date
raw = raw.rename(columns={raw.columns[0]: "DATE"})

old_cols = get_old_columns(raw)


# Testing get_lastest_data

In [ ]:
for download_date in ["2004-01-15", "2005-02-15", "2025-02-15"]:
    df = get_lastest_data(download_date)
    print(
        f"download_date={download_date} -> {len(df)}, rows, last date: {df['DATE'].max()}"
    )

## Start from empty Table

In [ ]:
with duckdb.connect(db_file) as con:
    for table in ["cpi_append", "cpi_trunc", "cpi_inc"]:
        con.execute(f"DROP TABLE IF EXISTS {table}")
        print(f" dropped{table}")
initialize_db()
print("Tables recreated empty")

## Daily Simulation Loop

In [ ]:
start = date(2004, 1, 1)
end = date(2005, 12, 31)
total_days = (end - start).days + 1

print(f"Simulating {total_days} days ({start} -> {end}) ...")

timings = {"append": [], "trunc": [], "inc": []}
dates = []

with duckdb.connect(db_file) as con:
    for i in range(total_days):
        pull_date = (start + timedelta(days=i)).strftime("%Y-%m-%d")
        dates.append(pull_date)

        t0 = time.perf_counter()
        con.execute("BEGIN TRANSACTION")
        append_data(con, pull_date)
        con.execute("COMMIT")
        timings["append"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        truncate_data(con, pull_date)
        timings["trunc"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        con.execute("BEGIN TRANSACTION")
        incremental_data(con, pull_date)
        con.execute("COMMIT")
        timings["inc"].append(time.perf_counter() - t0)

        if i % 100 == 0:
            print(f"  {pull_date} ...")

print("Done.")


## Speed Test

In [ ]:
timing_df = pd.DataFrame(timings, index=dates)

print("Mean time per run (ms):")
print((timing_df.mean() * 1000).round(2).to_string())
print()
print("Total time (seconds):")
print(timing_df.sum().round(3).to_string())

## Consistency Test